# n → ∞ 极限下 A-formulation 算法行为测试

**核心问题**：score ≈ 150 是 sampling noise 造成的，还是算法本身的 local minima？

**方法**：用理论协方差 `S_pop = (I-B*^T)^{-1} Ω* (I-B*)^{-1}` 代替样本协方差（等价于 n → ∞）。

**A formulation**：
- 目标函数：`f(A) = -2 log det(A) + tr(A^T S A)`
- 真实参数：`A_true = (I - B*) @ diag(1/sqrt(N))`
- population 下 f_true = `sum(log N_i) + d`（trace 项恰好等于 d）

**算法**：`dag_coordinate_descent_l0`（随机顺序）和 `dag_coordinate_descent_l0_epoch`（epoch）

**参数**：λ ∈ {0, 0.1, 0.2}，每组 100 次随机初始化（seed=0..99）

## Step 0: Imports & Setup

In [ ]:
import sys, os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from collections import Counter

sys.path.append(os.path.abspath(os.path.join(os.getcwd(), "..")))
sys.path.append(r"C:\Users\super\DAG")
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

from MEC import is_in_markov_equiv_class
from coordinate_descent.coordinate0 import (
    f,
    weight_to_adjacency,
    dag_coordinate_descent_l0,
    dag_coordinate_descent_l0_epoch,
)

print(f"Working directory: {os.getcwd()}")

## Step 1: 定义真实模型（B*, N）

沿用 test_CD_20251226 的 9 个实验。

> **Convention**：`N[i]` 是变量 i 的噪声方差（σ²_i）。
> 模型：`X = B*^T X + ε`，`ε_i ~ N(0, N_i)`。
> 真实参数：`A_true = (I - B*) @ diag(1/sqrt(N))`。

In [ ]:
experiments = []

# ----------- Experiment 1 -----------
experiments.append({
    "name": "d=3, A→B←C",
    "B_true": np.array([
        [0, 1, 0],
        [0, 0, 0],
        [0, 2, 0]
    ], dtype=float),
    "N": np.array([1, 2, 3], dtype=float),
})

# ----------- Experiment 2 -----------
experiments.append({
    "name": "d=3, A→B→C",
    "B_true": np.array([
        [0, 1, 0],
        [0, 0, 3],
        [0, 0, 0]
    ], dtype=float),
    "N": np.array([1, 3, 4], dtype=float),
})

# ----------- Experiment 3 -----------
experiments.append({
    "name": "d=3, A→B→C + A→C",
    "B_true": np.array([
        [0, 1, 2],
        [0, 0, 3],
        [0, 0, 0]
    ], dtype=float),
    "N": np.array([5, 4, 3], dtype=float),
})

# ----------- Experiment 4 -----------
experiments.append({
    "name": "d=4, A→B, B→C, B→D",
    "B_true": np.array([
        [0, 3, 0, 0],
        [0, 0, 3, 4],
        [0, 0, 0, 0],
        [0, 0, 0, 0]
    ], dtype=float),
    "N": np.array([1, 3, 3, 2], dtype=float),
})

# ----------- Experiment 5 -----------
experiments.append({
    "name": "d=4, A→C, A→D, B→C, B→D",
    "B_true": np.array([
        [0, 0, 2, 3],
        [0, 0, 3, 4],
        [0, 0, 0, 0],
        [0, 0, 0, 0]
    ], dtype=float),
    "N": np.array([2, 4, 3, 5], dtype=float),
})

# ----------- Experiment 6 -----------
experiments.append({
    "name": "d=4, A→D, B→D, C→D",
    "B_true": np.array([
        [0, 0, 0, 1],
        [0, 0, 0, 3],
        [0, 0, 0, 5],
        [0, 0, 0, 0]
    ], dtype=float),
    "N": np.array([5, 4, 3, 2], dtype=float),
})

# ----------- Experiment 7 -----------
experiments.append({
    "name": "d=5, e=4, |v|=0",
    "B_true": np.array([
        [0, 1, 0, 2, 0],
        [0, 0, 3, 0, 4],
        [0, 0, 0, 0, 0],
        [0, 0, 0, 0, 0],
        [0, 0, 0, 0, 0]
    ], dtype=float),
    "N": np.array([1, 2, 3, 2, 1], dtype=float),
})

# ----------- Experiment 8 -----------
experiments.append({
    "name": "d=5, e=4, |v|=1",
    "B_true": np.array([
        [0, 0, 1, 2, 0],
        [0, 0, 0, 2, 3],
        [0, 0, 0, 0, 0],
        [0, 0, 0, 0, 0],
        [0, 0, 0, 0, 0]
    ], dtype=float),
    "N": np.array([1, 2, 3, 2, 1], dtype=float),
})

# ----------- Experiment 9 -----------
experiments.append({
    "name": "d=5, e=4, |v|=2",
    "B_true": np.array([
        [0, 0, 0, 1, 0],
        [0, 0, 0, 2, 3],
        [0, 0, 0, 0, 4],
        [0, 0, 0, 0, 0],
        [0, 0, 0, 0, 0]
    ], dtype=float),
    "N": np.array([1, 2, 3, 2, 1], dtype=float),
})

print(f"Defined {len(experiments)} experiments")

## Step 2: 构造理论协方差 S_pop 与真实分数 f_true

$$S_{\text{pop}} = (I - B^{*T})^{-1} \, \text{diag}(N) \, (I - B^*)^{-1}$$

**A_true** 的构造（同 test_CD_20251226.ipynb）：
$$A^* = (I - B^*) \cdot \text{diag}(1/\sqrt{N})$$

**验证**：在真实参数处 `tr(A_true^T S_pop A_true) = d`，因此：
$$f^* = f(A^*, S_{\text{pop}}) = \sum_i \log N_i + d$$

这是 **n→∞ 下算法理论上能达到的最低分数下界**。

In [ ]:
def compute_S_pop(B_true: np.ndarray, N: np.ndarray) -> np.ndarray:
    """
    Population covariance: S_pop = (I - B^T)^{-1} diag(N) (I - B)^{-1}
    where N[i] = noise variance of variable i.
    """
    d = B_true.shape[0]
    M_inv = np.linalg.inv(np.eye(d) - B_true.T)  # (I - B^T)^{-1}
    return M_inv @ np.diag(N) @ M_inv.T


def compute_A_true(B_true: np.ndarray, N: np.ndarray) -> np.ndarray:
    """
    A_true = (I - B) @ diag(1/sqrt(N))
    Convention from test_CD_20251226: A_true = (I - B) @ inv(sqrtm(diag(N)))
    """
    d = B_true.shape[0]
    return (np.eye(d) - B_true) @ np.diag(1.0 / np.sqrt(N))


def true_score_population(B_true: np.ndarray, N: np.ndarray) -> float:
    """
    f(A_true, S_pop) = sum(log N_i) + d
    [since tr(A_true^T S_pop A_true) = d at the true parameters]
    """
    A_true = compute_A_true(B_true, N)
    S_pop  = compute_S_pop(B_true, N)
    return f(A_true, S_pop)


# Verify on all experiments
print(f"{'Experiment':<35} {'f_true (formula)':>18} {'f_true (computed)':>18} {'match':>6}")
print("-" * 82)
for exp in experiments:
    B = exp["B_true"]
    N = exp["N"]
    d = B.shape[0]
    f_formula  = float(np.sum(np.log(N)) + d)     # analytic: sum(log N) + d
    f_computed = true_score_population(B, N)
    match = "✓" if abs(f_formula - f_computed) < 1e-8 else "✗"
    print(f"{exp['name']:<35} {f_formula:>18.6f} {f_computed:>18.6f} {match:>6}")

## Step 3 & 4: 多次运行算法 + 分析 sparsity pattern

对每个实验、每个算法（random / epoch）、每个 λ（0, 0.1, 0.2），运行 100 次（seed=0..99）。

记录：Â[i]、G_hat[i]（sparsity pattern）、score[i]、is_mec[i]、score_gap = score - f_true。

In [ ]:
def adj_to_key(G: np.ndarray) -> str:
    return str(G.flatten().tolist())


def run_experiment_inf_A(
    exp: dict,
    n_repeats: int = 100,
    lambdas: list = [0.0, 0.1, 0.2],
    T_random: int = 5000,
    n_epochs: int = 500,
    threshold: float = 0.05,
) -> dict:
    """
    Run both random and epoch A-formulation algorithms on population covariance.
    Returns a nested dict: results[lambda][algo] = list of run dicts.
    """
    B_true  = exp["B_true"]
    N       = exp["N"]

    S_pop   = compute_S_pop(B_true, N)
    G_true  = weight_to_adjacency(B_true, threshold)
    f_true  = true_score_population(B_true, N)

    results = {}

    for lam in lambdas:
        results[lam] = {"random": [], "epoch": []}

        for seed in range(n_repeats):
            # ---- Random (non-epoch) ----
            A_hat, G_hat, score = dag_coordinate_descent_l0(
                S_pop, T=T_random, seed=seed,
                threshold=threshold, lambda_l0=lam,
            )
            is_mec = is_in_markov_equiv_class(G_true, G_hat)
            results[lam]["random"].append({
                "seed":      seed,
                "A_hat":     A_hat,
                "G_hat":     G_hat,
                "G_key":     adj_to_key(G_hat),
                "score":     score,
                "is_mec":    is_mec,
                "score_gap": score - f_true,
            })

            # ---- Epoch ----
            A_hat_e, G_hat_e, score_e, _ = dag_coordinate_descent_l0_epoch(
                S_pop, n_epochs=n_epochs, seed=seed,
                threshold=threshold, lambda_l0=lam,
                verbose=False,
            )
            is_mec_e = is_in_markov_equiv_class(G_true, G_hat_e)
            results[lam]["epoch"].append({
                "seed":      seed,
                "A_hat":     A_hat_e,
                "G_hat":     G_hat_e,
                "G_key":     adj_to_key(G_hat_e),
                "score":     score_e,
                "is_mec":    is_mec_e,
                "score_gap": score_e - f_true,
            })

    return {
        "name":      exp["name"],
        "B_true":    B_true,
        "G_true":    G_true,
        "S_pop":     S_pop,
        "f_true":    f_true,
        "results":   results,
    }

In [ ]:
# ============================================================
#  Run all experiments  (this may take a few minutes)
# ============================================================
LAMBDAS   = [0.0, 0.1, 0.2]
N_REPEATS = 100
T_RANDOM  = 5000
N_EPOCHS  = 500
THRESHOLD = 0.05

all_results = []

for idx, exp in enumerate(experiments):
    print(f"[{idx+1}/{len(experiments)}] {exp['name']} ...", flush=True)
    res = run_experiment_inf_A(
        exp,
        n_repeats=N_REPEATS,
        lambdas=LAMBDAS,
        T_random=T_RANDOM,
        n_epochs=N_EPOCHS,
        threshold=THRESHOLD,
    )
    all_results.append(res)
    # Quick summary
    for lam in LAMBDAS:
        for algo in ["random", "epoch"]:
            runs = res["results"][lam][algo]
            mec_rate   = np.mean([r["is_mec"] for r in runs])
            n_patterns = len(set(r["G_key"] for r in runs))
            mean_score = np.mean([r["score"] for r in runs])
            print(f"  λ={lam}  {algo:6s}: MEC={mec_rate:.2f}  "
                  f"patterns={n_patterns:2d}  mean_score={mean_score:.4f}  "
                  f"f_true={res['f_true']:.4f}")

print("\nDone.")

## Analysis: Summary Table

In [ ]:
rows = []
for res in all_results:
    for lam in LAMBDAS:
        for algo in ["random", "epoch"]:
            runs   = res["results"][lam][algo]
            scores = [r["score"] for r in runs]
            gaps   = [r["score_gap"] for r in runs]
            rows.append({
                "experiment": res["name"],
                "lambda":     lam,
                "algorithm":  algo,
                "MEC_rate":   round(np.mean([r["is_mec"] for r in runs]), 2),
                "n_patterns": len(set(r["G_key"] for r in runs)),
                "f_true":     round(res["f_true"], 4),
                "score_mean": round(np.mean(scores), 4),
                "score_min":  round(np.min(scores), 4),
                "score_max":  round(np.max(scores), 4),
                "gap_mean":   round(np.mean(gaps), 4),
                "gap_max":    round(np.max(gaps), 4),
            })

df = pd.DataFrame(rows)
pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 150)
print(df.to_string(index=False))

## Analysis: Sparsity Pattern Frequency

In [ ]:
def summarize_patterns(res: dict, lam: float, algo: str, top_k: int = 5) -> None:
    runs    = res["results"][lam][algo]
    counter = Counter(r["G_key"] for r in runs)

    seen = {}
    for r in runs:
        k = r["G_key"]
        if k not in seen:
            seen[k] = {"G": r["G_hat"], "scores": [], "mec": r["is_mec"]}
        seen[k]["scores"].append(r["score"])

    print(f"  G_true:    {res['G_true'].flatten().tolist()}")
    print(f"  f_true:    {res['f_true']:.4f}")
    print(f"  Top-{top_k} patterns  (freq | is_mec | mean_score | G_flat):")
    for k in sorted(seen, key=lambda x: -counter[x])[:top_k]:
        freq   = counter[k]
        is_mec = seen[k]["mec"]
        mscore = np.mean(seen[k]["scores"])
        gflat  = seen[k]["G"].flatten().tolist()
        tag    = " ← TRUE MEC" if is_mec else ""
        print(f"    freq={freq:3d}  mec={str(is_mec):<5}  score={mscore:.4f}  {gflat}{tag}")


print("=" * 80)
print("SPARSITY PATTERN ANALYSIS  (A-formulation, n→∞, 100 runs each)")
print("=" * 80)

for res in all_results:
    print(f"\n{'='*60}")
    print(f"Experiment: {res['name']}")
    for lam in LAMBDAS:
        for algo in ["random", "epoch"]:
            print(f"\n  λ={lam}  algorithm={algo}")
            summarize_patterns(res, lam, algo)

## Analysis: Score vs Structure (is_mec)

In [ ]:
from matplotlib.patches import Patch

def plot_score_vs_struct(res: dict, lam: float) -> None:
    fig, axes = plt.subplots(1, 2, figsize=(14, 4), sharey=False)
    fig.suptitle(f"{res['name']}  |  λ={lam}  (A-formulation, n→∞)", fontsize=12)

    for ax, algo in zip(axes, ["random", "epoch"]):
        runs   = res["results"][lam][algo]
        seeds  = [r["seed"]  for r in runs]
        scores = [r["score"] for r in runs]
        colors = ["#2196F3" if r["is_mec"] else "#F44336" for r in runs]

        ax.scatter(seeds, scores, c=colors, s=18, alpha=0.8, zorder=3)
        ax.axhline(res["f_true"], color="black", linestyle="--", linewidth=1.5,
                   label=f"f_true={res['f_true']:.3f}")

        mec_rate = np.mean([r["is_mec"] for r in runs])
        ax.set_title(f"{algo}  (MEC={mec_rate:.2f})", fontsize=10)
        ax.set_xlabel("seed")
        ax.set_ylabel("f(A)")

        legend_elems = [
            Patch(facecolor="#2196F3", label="is_mec=True"),
            Patch(facecolor="#F44336", label="is_mec=False"),
            plt.Line2D([0], [0], color="black", linestyle="--",
                       label=f"f_true={res['f_true']:.3f}"),
        ]
        ax.legend(handles=legend_elems, fontsize=8, loc="upper right")

    plt.tight_layout()
    plt.show()


for res in all_results:
    for lam in LAMBDAS:
        plot_score_vs_struct(res, lam)

## Analysis: Score Distribution Histogram (is_mec=True vs False)

In [ ]:
def plot_score_histogram(res: dict, lam: float) -> None:
    fig, axes = plt.subplots(1, 2, figsize=(14, 4))
    fig.suptitle(f"{res['name']}  |  λ={lam}  — Score Histogram  (A-formulation, n→∞)",
                 fontsize=11)

    for ax, algo in zip(axes, ["random", "epoch"]):
        runs  = res["results"][lam][algo]
        sc_t  = [r["score"] for r in runs if r["is_mec"]]
        sc_f  = [r["score"] for r in runs if not r["is_mec"]]

        all_scores = [r["score"] for r in runs]
        bins = np.linspace(min(all_scores) - 0.05, max(all_scores) + 0.05, 25)

        if sc_t:
            ax.hist(sc_t, bins=bins, color="#2196F3", alpha=0.7,
                    label=f"is_mec=True (n={len(sc_t)})")
        if sc_f:
            ax.hist(sc_f, bins=bins, color="#F44336", alpha=0.7,
                    label=f"is_mec=False (n={len(sc_f)})")
        ax.axvline(res["f_true"], color="black", linestyle="--", linewidth=1.5,
                   label=f"f_true={res['f_true']:.3f}")
        ax.set_title(f"{algo}", fontsize=10)
        ax.set_xlabel("f(A)")
        ax.set_ylabel("count")
        ax.legend(fontsize=8)

    plt.tight_layout()
    plt.show()


for res in all_results:
    for lam in LAMBDAS:
        plot_score_histogram(res, lam)

## Analysis: λ 对 MEC rate 的影响

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("MEC Rate vs λ  (A-formulation, n→∞, 100 seeds)", fontsize=12)

x = np.arange(len(LAMBDAS))
width = 0.08
colors = plt.cm.tab10(np.linspace(0, 0.9, len(experiments)))

for ax, algo in zip(axes, ["random", "epoch"]):
    for i, res in enumerate(all_results):
        mec_rates = [
            np.mean([r["is_mec"] for r in res["results"][lam][algo]])
            for lam in LAMBDAS
        ]
        offset = (i - len(experiments) / 2) * width
        ax.bar(x + offset, mec_rates, width=width, color=colors[i],
               label=res["name"], alpha=0.85)

    ax.set_title(f"algorithm = {algo}", fontsize=10)
    ax.set_xticks(x)
    ax.set_xticklabels([f"λ={l}" for l in LAMBDAS])
    ax.set_ylabel("MEC rate")
    ax.set_ylim(0, 1.05)
    ax.axhline(1.0, color="gray", linestyle=":", linewidth=1)
    ax.legend(fontsize=6, loc="upper right", ncol=2)

plt.tight_layout()
plt.show()

## Analysis: Score Gap  (f_hat - f_true)

- `gap ≈ 0`：算法收敛到全局最优
- `gap > 0`：local minima（算法本身的问题，与 sampling noise 无关）
- `gap < 0`：不应发生（A_true 是理论最优）

In [ ]:
print(f"{'Experiment':<35} {'algo':>6} {'lam':>5}  "
      f"{'gap_mean':>10} {'gap_max':>10} {'gap_std':>10}  "
      f"{'n_gap>0.01':>10}")
print("-" * 100)

for res in all_results:
    for algo in ["random", "epoch"]:
        for lam in LAMBDAS:
            runs = res["results"][lam][algo]
            gaps = np.array([r["score_gap"] for r in runs])
            n_pos = int(np.sum(gaps > 0.01))
            print(f"{res['name']:<35} {algo:>6} {lam:>5.1f}  "
                  f"{gaps.mean():>10.4f} {gaps.max():>10.4f} {gaps.std():>10.4f}  "
                  f"{n_pos:>10}")

## 与 B_Omega formulation 对比

对每个实验，对比两种 formulation 在 n→∞ 下的 MEC rate（λ=0）。

In [ ]:
# MEC rate table at lambda=0
lam = 0.0
print(f"{'Experiment':<35} {'random MEC':>12} {'epoch MEC':>12}   (A-formulation, λ=0, n→∞)")
print("-" * 65)
for res in all_results:
    r_mec = np.mean([r["is_mec"] for r in res["results"][lam]["random"]])
    e_mec = np.mean([r["is_mec"] for r in res["results"][lam]["epoch"]])
    print(f"{res['name']:<35} {r_mec:>12.2f} {e_mec:>12.2f}")

## 结论

**核心问题回答**：
- 若 n→∞ 下 `score_gap > 0`（即 f_hat > f_true），说明高 score 来自**算法 local minima**，而非 sampling noise。
- 若 `score_gap ≈ 0` 但 `is_mec=False`，说明存在等价 score 的错误结构（MEC 内的非等价图）。

**与 B_Omega formulation 对比**：
- 两种 formulation 的目标函数在数学上等价（`f(A) = 2 * ell(B, Ω)`），但优化 landscape 不同。
- 比较 MEC rate 和 score gap，可以判断哪种 parameterization 更容易陷入 local minima。